https://spark.apache.org/docs/latest/sql-getting-started.html

# Intialisation de spark (par défaut activer)

Recherche des caractéristiques de l'ordinateur pour configurer Spark selon les performances de la machine disponible

In [70]:
import org.apache.spark.sql.SparkSession

val spark = SparkSession.builder()
  .appName("Projet_scala")
  .master("local[*]")                // local[*] pour tous
  .config("spark.executor.memory", "1g") // mémoire pour chaque exécuteur
  .config("spark.driver.memory", "1g")   // mémoire pour le driver
  .config("spark.executor.cores", "3") \
  .config("spark.sql.shuffle.partitions", "4") // nombre de partitions pour shuffle
  .getOrCreate()

val sc = spark.sparkContext

// Récupération des infos de config avec sécurité (getOption évite les erreurs si la clé n'existe pas)
val master        = sc.getConf.get("spark.master")
val driverMem     = sc.getConf.getOption("spark.driver.memory").getOrElse("non défini")
val executorMem   = sc.getConf.getOption("spark.executor.memory").getOrElse("non défini")
val executorCores = sc.getConf.getOption("spark.executor.cores").getOrElse("non défini")
val parallelism   = sc.defaultParallelism
val shuffleParts  = sc.getConf.getOption("spark.sql.shuffle.partitions").getOrElse("par défaut (200)")

// Affichage lisible
println("===== Configuration Spark détectée =====")
println(f"Master                 : $master")
println(f"Driver Memory          : $driverMem")
println(f"Executor Memory        : $executorMem")
println(f"Executor Cores         : $executorCores")
println(f"Default Parallelism    : $parallelism")
println(f"Shuffle Partitions SQL : $shuffleParts")
println("========================================")


===== Configuration Spark détectée =====
Master                 : local[*]
Driver Memory          : 1g
Executor Memory        : 1g
Executor Cores         : non défini
Default Parallelism    : 3
Shuffle Partitions SQL : 4


lastException = null
spark = org.apache.spark.sql.SparkSession@79a52cca
sc = org.apache.spark.SparkContext@78fe4cb2
master = local[*]
driverMem = 1g
executorMem = 1g
executorCores = non défini
parallelism = 3
shuffleParts = 4


4

In [49]:
// contôle de la version de Spark
sc.version

3.5.0

# Chargement des données Google
https://snap.stanford.edu/data/web-Google.txt.gz

Données présentes en locale dans le dossier du script

In [2]:
// Import des données pas de modif possible
val df = spark.read
.options(Map("delimiter" -> "\t", "header" -> "false", "comment" -> "#")) // séparateur tabulation, retrait des commentaires, pas de titre
.csv("web-Google.txt") // fichier à ouvrir
.toDF("n0", "n1") // non de colonne
.withColumn("n0", $"n0".cast("Int")) // définition du type pour chaque colonne
.withColumn("n1", $"n1".cast("Int"))

// println(df.printSchema())
// println(df.count())
println(df.getClass())
df.show(5)

class org.apache.spark.sql.Dataset
+-----+------+
|   n0|    n1|
+-----+------+
|    0| 11342|
|    0|824020|
|    0|867923|
|    0|891835|
|11342|     0|
+-----+------+
only showing top 5 rows



df = [n0: int, n1: int]


[n0: int, n1: int]

In [4]:
// vérification du nombre de partition de HDFS du stockage du dataframe
val inputFiles = df.inputFiles
println("Fichiers HDFS utilisés : " + inputFiles.length)
inputFiles.foreach(println)

inputFiles = Array(file:///data/indir/Module_3_donnees_massives/Traitement_distribué/Projet_Scala_hadoop/web-Google.txt)


Fichiers HDFS utilisés : 1
file:///data/indir/Module_3_donnees_massives/Traitement_distribué/Projet_Scala_hadoop/web-Google.txt


Array(file:///data/indir/Module_3_donnees_massives/Traitement_distribué/Projet_Scala_hadoop/web-Google.txt)

# DataFrame

Version Dataframe mal parallélisée, car présence de collect qui agrége les données 

### Création des fonctions CCF pour DataFrame 
ccf_iterate : parcours de l'ensemble des valeurs pour définir l'élément le plus faible + export des liens entre deux noeuds

ccf_iterate_sort : tri des valeurs pour extraire à la première position l'élément le plus faible + export des liens entre deux noeuds

In [42]:
import scala.collection.mutable.ArrayBuffer

// fonction ccf avec compteur de nouvelles paires
def ccf_iterate(key: Int, assoc: Array[Int], data_df: ArrayBuffer[(Int, Int)]): Int = {
  var mini: Int = key
  var newPair = 0 // intialisation du compteur des nouvelles paires pour réaliser la boucle while
  // parcours de l'ensemble des valeurs pour définir la plus petite valeur existante entre la clé et la collection de valeurs
  for (valeur <- assoc if valeur < mini) mini = valeur
  // si une valeur plus petite que la clé existe
  if (mini < key) {
    // ajout de la paire clé -> mini (non comptée dans newPair)
    data_df += ((key, mini))
    // ajout des autres paires et comptage
    for (valeur <- assoc if valeur != mini) {
        data_df += ((valeur, mini))
        newPair += 1
    }
  }
  newPair // retour new count pour définir si besoin d'une nouvelle itération
}

// fonction ccf avec les valeurs triées + présence du compteur de nouvelles paires
def ccf_iterate_sort(key:Int, assoc:Array[Int], data_df:ArrayBuffer[(Int, Int)]): Int = {
    val tri:Array[Int] = assoc.sorted // tri des valeurs présentes dans l'array
    var mini:Int = tri.head // extraction du 1er élément qui est le plus petit élément
    val autre: Array[Int] = tri.drop(1) // et suppression du 1er élément 
    var newPair = 0 // intialisation du compteur des nouvelles paires pour réaliser la boucle while 
    // si une valeur plus petite que la clé existe
    if (mini < key) {
    // ajout de la paire clé -> mini (non comptée dans newPair)
        data_df += ((key, mini))
        // ajout des autres paires et comptage
        for (valeur <- autre ) {
            data_df += ((valeur, mini))
            newPair += 1
            }
        }
    newPair // retour new count pour définir si besoin d'une nouvelle itération
}

ccf_iterate: (key: Int, assoc: Array[Int], data_df: scala.collection.mutable.ArrayBuffer[(Int, Int)])Int
ccf_iterate_sort: (key: Int, assoc: Array[Int], data_df: scala.collection.mutable.ArrayBuffer[(Int, Int)])Int


### Création d'une fonction runCCF pour DataFrame
Appel de la fonction avec choix d'options :

- taille du dataset à tester (taille_ech)

- type d'algorithme à utiliser, avec selection de ccf_iterate ou ccf_iterate_sort (ccf)

In [6]:
// fonction pour appeler selon les deux algoritmes et différentes taille de dataset

import org.apache.spark.sql.{Dataset, Row, DataFrame}   
import java.io.{File, PrintWriter}

def runCCF( df: Dataset[Row], taille_ech: Int, ccf: (Int, Array[Int], ArrayBuffer[(Int, Int)]) => Int , param_deduplic:String, method:String, pw: PrintWriter ): Unit = {
    // Récupération et dimentionnement des données initiales
    val data_df: ArrayBuffer[(Int, Int)] = ArrayBuffer(df.limit(taille_ech).collect().map(row => (row.getInt(0), row.getInt(1))): _*) // : _* est nécessaire pour "dépackager" l’Array en arguments pour le constructeur d’ArrayBuffer.
    
    // initialisation pour stockage des paires dédoublés
    val globalAB = ArrayBuffer[(Int, Int)]()     
    // mesure des boucles itératives
    var compteur = 1 // compteur des nouvelles paires intialisé à un pour entrer dans la boucle while
    var tour = 0 // compteur pour définir le nombre d'itération  
    val startTime = System.nanoTime() //mesure du temps d'execution 
    
    while (compteur > 0) {
        tour += 1 // compte le nombre de tour
        compteur = 0 // réinitialisation du compteur de nouvelles paires, si incrémentation besoin d'une nouvelle itération
        
        // initialisation de globalAB avant ajout des paires bidirectionnelles
        globalAB.clear() 
        // selon l'algorithme choisi suppression ou non des paires identiques
        if (param_deduplic == "no") {
            globalAB ++= data_df.flatMap { case (n0, n1) => Seq((n0, n1), (n1, n0)) } }
        if (param_deduplic == "yes") {
            // distinc pour faire CCF-dedup de l'article
            globalAB ++= data_df.distinct.flatMap { case (n0, n1) => Seq((n0, n1), (n1, n0)) } }
        
        // réalisation du shuffle and sort avec construction de type clé [Int] et collection de valeurs Array[Int]
        val group = globalAB.groupBy(_._1).map { case (k, arr) => (k, arr.map(_._2).toArray) }
        
        // Réalisation du map et de la transformation des paires par appel de la fonction ccf-iterate ou ccf-iterate-sort
        data_df.clear() // réinitialisation de data_df avant la fonction ccf pour rechercher les nouveaux couples entre deux noeuds
        // addition du retour de newPair de chaque appel de ccf au coumpteur avec sum, ce qui permet également la mise à jours de data_df avec les nouvelles paires
        compteur = group.map { case (k, arr) => ccf(k, arr, data_df) }.sum 
        // println(data_df.mkString(", "))
} 

val endTime = System.nanoTime()
val tempsec = (endTime - startTime) / 1e9 // calcul du temps d'execution en seconde
val nb_groupe =  data_df.map(_._2).distinct.size // (_._2), recherche uniquement dans la seconde colonne pour définir le nombre de groupe
println(f"Test ${method} avec ${taille_ech} liens, temps écoulé : $tempsec%.3f secondes")
// println(s"Nombre total de paires : ${data_df.size}")
// println(s"Nombre de groupe : ${nb_groupe}")
// println(data_df.mkString(", "))

// export des résultats dans un fichier
pw.println(Seq(method,taille_ech,data_df.size,nb_groupe,tour,1,tempsec).mkString(","))

}


runCCF: (df: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row], taille_ech: Int, ccf: (Int, Array[Int], scala.collection.mutable.ArrayBuffer[(Int, Int)]) => Int, param_deduplic: String, method: String, pw: java.io.PrintWriter)Unit


Création d'une liste pour définir les tailles du dataset a tester

In [7]:
{ 
// Création d'un array à parcourir pour faire les tests de ccf, limitation pas ensemble des données
val repet:Int = 3
val test_vol_data: Array[Int] = 
  Array.fill(repet)(10) ++   // L pour type Long et pas Int compatible avec le type d'array
  Array.fill(repet)(100) ++  
  Array.fill(repet)(1000) ++ 
  Array.fill(repet)(10000) ++ 
  Array.fill(repet)(100000) ++ 
  Array.fill(repet)(500000) // ++ 
  // Array.fill(1)(1000000) ++ // pas de répétition sur les grands jeu de données variation négligeable
  // Array.fill(1)(2000000) ++ 
  // Array.fill(1)(df.count().toInt)
}

repet = 3
test_vol_data = Array(10, 10, 10, 100, 100, 100, 1000, 1000, 1000, 10000, 10000, 10000, 100000, 100000, 100000, 500000, 500000, 500000)


Array(10, 10, 10, 100, 100, 100, 1000, 1000, 1000, 10000, 10000, 10000, 100000, 100000, 100000, 500000, 500000, 500000)

### Execution de la function runCCF avec comme varaible : 
- df : le tableau complet de données Google
- n : nombre d'arrêtes sélectionnées pour le test
- ccf : choix de l'algoritme
- param_deduplic : "yes ou "no" pour définir si utilisation de CCF-dedup
- method : Legende du test fait
- pw : fichier pour enregistre les résultats du test

In [8]:
// Execution des tests et sauvegarde des résultats 

// Création du writer (mode écriture) // ATTENTION : ça écrase le fichier si déjà existant
val pw = new PrintWriter(new File("Scala_DF.csv"))
// Écrire l’en-tête (une seule fois avant la boucle)
pw.println(Seq("methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps").mkString(","))
// Test et écriture des résultats
for ( n <- test_vol_data ) { runCCF(df, n, ccf_iterate , "no" , "df_scala_iterate_avec_duplicat", pw) } 
for ( n <- test_vol_data ) { runCCF(df, n, ccf_iterate , "yes" , "df_scala_iterate_sans_duplicat", pw) } 
for ( n <- test_vol_data ) { runCCF(df, n, ccf_iterate_sort , "yes" , "df_scala_iterate_sort_sans_duplicat", pw) } 
// Fermer le writer à la fin
pw.close() // Ne fermer ici si ajout des résultats RDD dans ce même fichier

Test df_scala_iterate_avec_duplicat avec 10 liens, temps écoulé : 0.005 secondes
Test df_scala_iterate_avec_duplicat avec 10 liens, temps écoulé : 0.001 secondes
Test df_scala_iterate_avec_duplicat avec 10 liens, temps écoulé : 0.001 secondes
Test df_scala_iterate_avec_duplicat avec 100 liens, temps écoulé : 0.005 secondes
Test df_scala_iterate_avec_duplicat avec 100 liens, temps écoulé : 0.003 secondes
Test df_scala_iterate_avec_duplicat avec 100 liens, temps écoulé : 0.002 secondes
Test df_scala_iterate_avec_duplicat avec 1000 liens, temps écoulé : 0.012 secondes
Test df_scala_iterate_avec_duplicat avec 1000 liens, temps écoulé : 0.016 secondes
Test df_scala_iterate_avec_duplicat avec 1000 liens, temps écoulé : 0.016 secondes
Test df_scala_iterate_avec_duplicat avec 10000 liens, temps écoulé : 0.361 secondes
Test df_scala_iterate_avec_duplicat avec 10000 liens, temps écoulé : 0.252 secondes
Test df_scala_iterate_avec_duplicat avec 10000 liens, temps écoulé : 0.125 secondes
Test df_sc

pw = java.io.PrintWriter@2aa08499


java.io.PrintWriter@2aa08499

Utilisation de chatGPT pour optimiser la parallélisation des DataFrames, réduction de collect. Temps plus long que le premier développement et toujours pas de résultat pour le fichier complet (résultat non montré dans le rapport)

In [71]:
import scala.collection.mutable.WrappedArray

// ------------------------------
// 1. UDF pour CCF
// ------------------------------
val ccfUDF = udf((key: Int, assoc: WrappedArray[Int]) => {
  if (assoc.isEmpty) Seq.empty[(Int, Int)]
  else {
    val tri = assoc.sorted
    val mini = tri.head
    val autre = tri.drop(1)
    if (mini < key) {
      ((key, mini) +: autre.map(v => (v, mini))).toSeq
    } else Seq.empty[(Int, Int)]
  }
})

// ------------------------------
// 2. Fonction runCCF DataFrame
// ------------------------------
def runCCF(
    df: Dataset[Row],
    taille_ech: Int,
    param_deduplic: String,
    method: String,
    pw: PrintWriter
): Unit = {

  val spark = df.sparkSession
  import spark.implicits._

  // 1️⃣ échantillon
  var current = df.limit(taille_ech).coalesce(3) 

  // 2️⃣ compteur et itération
  var compteur = 1
  var tour = 0
  var maxPartitions = 0  // variable pour stocker le max
  val startTime = System.nanoTime()
    

  while (compteur > 0) {
    tour += 1

    // 3️⃣ création des paires bidirectionnelles
    val edges = current.union(current.select(col("n1").as("n0"), col("n0").as("n1")))

    // 4️⃣ dédoublonnage si demandé
    val edges_dedup = if (param_deduplic == "yes") edges.distinct() else edges

    // 5️⃣ regroupement par clé (n0) et collect_list des voisins
    val grouped = edges_dedup.groupBy("n0")
      .agg(collect_list("n1").as("neighbors"))
    // println("Partitions après groupBy : " + grouped.rdd.getNumPartitions)
    // récupérer le nombre de partitions après le groupBy
    val nbPartitions = grouped.rdd.getNumPartitions
    maxPartitions = math.max(maxPartitions, nbPartitions)  // garder le max
    // println(s"Iteration $tour : partitions utilisées = $nbPartitions")

    // 6️⃣ application du CCF via UDF
    val new_pairs = grouped
      .withColumn("pairs", explode(ccfUDF(col("n0"), col("neighbors"))))
      .select(col("pairs._1").as("n0"), col("pairs._2").as("n1"))
    // println(new_pairs.show())

    // 7️⃣ décompte pour savoir si continuer l’itération
    // ⚠️ COMPTEUR = nombre de paires vraiment nouvelles
    val new_only = new_pairs.distinct.except(current) // toutes les paires qui n'étaient pas encore dans les aata avant l'iteration
    compteur = new_only.count().toInt

    // juste après avoir créé new_pairs
    var new_pairs_partitioned = new_pairs.coalesce(3)  // 3 partitions exactes
    current = new_pairs_partitioned
    // current = new_pairs //.groupBy("n1").agg(collect_list("n0").as("neighbors"))     
  }

  val endTime = System.nanoTime()
  val tempsec = (endTime - startTime) / 1e9

  // 8️⃣ nombre de groupes finaux
  val nb_groupe = current.distinct().count()

  println(f"Test $method avec $taille_ech liens, parallele $maxPartitions, temps écoulé : $tempsec%.3f secondes")
  // suppose df est ton DataFrame
  // val nbPartitions = new_pairs.rdd.getNumPartitions
  // println(s"Nombre de partitions : $nbPartitions")

  // 9️⃣ export vers le CSV
  pw.println(Seq(method, taille_ech, current.count(), nb_groupe, tour, maxPartitions, tempsec).mkString(","))
}

// ------------------------------
// 3. Execution des tests
// ------------------------------

val repet:Int = 3
val test_vol_data: Array[Int] = 
  Array.fill(repet)(10) ++   // L pour type Long et pas Int compatible avec le type d'array
  Array.fill(repet)(100) ++  
  Array.fill(repet)(1000) ++ 
  Array.fill(repet)(10000) ++ 
  Array.fill(repet)(100000) //++ 
  // Array.fill(repet)(500000) // ++ 
  // Array.fill(1)(1000000) ++ // pas de répétition sur les grands jeu de données variation négligeable
  // Array.fill(1)(2000000) ++ 
  // Array.fill(1)(df.count().toInt)

val pw = new PrintWriter(new File("/data/indir/Scala_DF.csv"))
pw.println(Seq("methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps").mkString(","))

// for (n <- test_vol_data) runCCF(df, n, "no", "df_scala_iterate_avec_duplicat", pw)
for (n <- test_vol_data) runCCF(df, n, "yes", "df_scala_iterate_sans_duplicat_code_parallele", pw)
// for (n <- test_vol_data) runCCF(df, n, "yes", "df_scala_iterate_sort_sans_duplicat", pw)

pw.close()


Test df_scala_iterate_sans_duplicat_code_parallele avec 10 liens, parallele 1, temps écoulé : 1.763 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 10 liens, parallele 1, temps écoulé : 1.913 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 10 liens, parallele 1, temps écoulé : 2.024 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 100 liens, parallele 1, temps écoulé : 2.057 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 100 liens, parallele 1, temps écoulé : 1.912 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 100 liens, parallele 1, temps écoulé : 1.973 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 1000 liens, parallele 1, temps écoulé : 2.533 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 1000 liens, parallele 1, temps écoulé : 2.787 secondes
Test df_scala_iterate_sans_duplicat_code_parallele avec 1000 liens, parallele 1, temps écoulé : 2.550 secondes
Test df_sc

ccfUDF = SparkUserDefinedFunction($Lambda$7201/0x00007e62a57d8a20@61a09dbc,ArrayType(StructType(StructField(_1,IntegerType,false),StructField(_2,IntegerType,false)),true),List(Some(class[value[0]: int]), Some(class[value[0]: array<int>])),Some(class[value[0]: array<struct<_1:int,_2:int>>]),None,true,true)
repet = 3
test_vol_data = Array(10, 10, 10, 100, 100, 100, 1000, 1000, 1000, 10000, 10000, 10000, 100000, 100000, 100000)
pw = java.io.PrintWriter@6847e84b


runCCF: (df: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row], taille_ech: Int, param_deduplic: String, method: String, pw: java.io.PrintWriter)Unit


java.io.PrintWriter@6847e84b

# RDD

### Création des fonctions CCF pour RDD 
ccf_iterate_RDD : parcours de l'ensemble des valeurs pour définir l'élément le plus faible + export des liens entre deux noeuds

ccf_iterate_sort_RDD : tri des valeurs pour extraire à la première position l'élément le plus faible + export des liens entre deux noeuds

### Création d'une fonction runCCF pour RDD
Appel de la fonction avec 3 options possibles : 

- Taille du dataset à tester (taile_ech)

- Type d'algorithme à utiliser, avec de ccf_iterate_rdd ou ccf_iterate_sort_rdd (ccf)

- Nombre de partition pour la parallélisation (param_parall)

In [38]:
import org.apache.spark.sql.{Dataset, Row, DataFrame}   
import java.io.{File, PrintWriter}

import org.apache.spark.rdd.RDD
import org.apache.spark.util.LongAccumulator
 
// fonction ccf adaptée pour RDD : retourne un Array[(Int, Int)]
// au lieu du retour du compteur comme pour les dataframes, utilisation d'un accumulateur ce qui permet le retour directement de l'array
// si traitement par les fonctions, problème -> passage par object
object CCFUtils {
    def ccf_iterate_RDD(key: Int, assoc: Array[Int], acc: LongAccumulator): Array[(Int, Int)] = {
        var mini: Int = key
        var newPairs = Array[(Int, Int)]() // nouvel Array pour enregistrer les nouvelles paires 
          // parcours de l'ensemble des valeurs pour définir la plus petite valeur existante entre la clé et la collection de valeurs
        for (valeur <- assoc if valeur < mini) mini = valeur
        if (mini < key) {
            // ajout de la paire clé -> mini
            newPairs :+= (key, mini) // ":" sert à ajouter un élément à la fin d’une collection mutable.
            // ajout des autres paires
            for (valeur <- assoc if valeur != mini) {
                newPairs :+= (valeur, mini)
                acc.add(1)  // incrémentation de l'accumulateur
            }
        }
        newPairs //return les nouvelles paires
    }
}

// fonction ccf adaptée pour RDD avec tri des données
object CCF2Utils {
    def ccf_iterate_sort_RDD(key: Int, assoc: Array[Int], acc: LongAccumulator): Array[(Int, Int)] = {
        val tri:Array[Int] = assoc.sorted //tri de l'array
        var mini:Int = tri.head // extraction du 1er élément
        val autre: Array[Int] = tri.drop(1) // et suppression du 1er élément 
        var newPairs = Array[(Int, Int)]() // nouvel array pour enregistrer les nouvelles paires 
    
        if (mini < key) {
            // ajout de la paire clé -> mini
            newPairs :+= (key, mini) // ":" sert à ajouter un élément à la fin d’une collection mutable.
            // ajout des autres paires
            for (valeur <- autre ) {
                newPairs :+= (valeur, mini)
                acc.add(1)  // incrémentation de l'accumulateur
            }
        }
        newPairs //return les nouvelles paires
    }
}

// fonction complète qui appele le choix de l'algorithme et permet de lancement des tests 
def runCCF_RDD( df: Dataset[Row], taille_ech: Int, ccfRDD: (Int, Array[Int], LongAccumulator ) => Array[(Int, Int)] , param_parall:Int, param_deduplic:String, method:String, pw: PrintWriter ): Unit = {
    // Création de l'accumulateur
    var compteurAcc = spark.sparkContext.longAccumulator("Compteur nouvelles paires")
    // Création RDD à partir d'un tableau 
    val data_df = df.limit(taille_ech).collect().map(row => (row.getInt(0), row.getInt(1)))
    var data_rdd = spark.sparkContext.parallelize(data_df, param_parall )
    // boucle itérative
    var compteur = 1
    var tour = 0
    
    val startTime = System.nanoTime()
    while (compteur > 0) {
        tour += 1
        compteurAcc.reset() // a chaque boucle réinitiialisation du compteur de nouvelle paires
        
        // initialisation pour stocker les paires bidirectionnelles avec +/- des duplicats
        val globalRDD = if (param_deduplic == "no") {
            data_rdd.flatMap { case (n0, n1) => Seq((n0, n1), (n1, n0)) }
        } else {
        // si param_deduplic == "yes"
            data_rdd.distinct().flatMap { case (n0, n1) => Seq((n0, n1), (n1, n0)) }
        }
        
        // groupement par clé et collection de valeurs
        val groupedRDD: RDD[(Int, Array[Int])] = globalRDD.groupByKey().map { case (k, iter) => (k, iter.toArray) }

        // application de ccfRDD sur chaque clé
        val newPairsRDD: RDD[(Int, Int)] = groupedRDD.flatMap { case (k, arr) => CCFUtils.ccf_iterate_RDD(k, arr, compteurAcc) }
    
        var tmp = newPairsRDD.count() // Action pour mettre à jours newPairsRDD et les différents éléments avant de faire une nouvelle itération
        compteur = compteurAcc.value.toInt 
        // println(s"Tour $tour : nouvelles paires = $compteur")
        
        data_rdd = newPairsRDD // action si dessus avec count, donc newPairs est à jours pour le tour suivant 

    }
    
    // Résultats finaux
    val endTime = System.nanoTime()
    val tempsec = (endTime - startTime) / 1e9 // calcul du temps d'execution en seconde
    
    val nb_groupe = data_rdd.map(_._2).distinct().count()
    println(f"Test ${method} avec ${taille_ech} liens et parallélisation ${param_parall}, temps écoulé : $tempsec%.3f secondes")
    // println(data_rdd.collect().mkString(", "))

    // export des résultats dans un fichier
    pw.println(Seq(method,taille_ech,data_rdd.count(),nb_groupe,tour,param_parall,tempsec).mkString(","))
}

defined object CCFUtils
defined object CCF2Utils


runCCF_RDD: (df: org.apache.spark.sql.Dataset[org.apache.spark.sql.Row], taille_ech: Int, ccfRDD: (Int, Array[Int], org.apache.spark.util.LongAccumulator) => Array[(Int, Int)], param_parall: Int, param_deduplic: String, method: String, pw: java.io.PrintWriter)Unit


### Test impact de la parallélisation sur le temps de traitement

In [16]:
// Test de parallélisation 
var test_nb_paralle = ArrayBuffer[Int]() 
test_nb_paralle ++= (1 to 30).toArray
test_nb_paralle ++= (1 to 30).toArray
test_nb_paralle ++= (1 to 30).toArray

// Création du writer (mode écriture) // ATTENTION : ça écrase le fichier si déjà existant
val pw = new PrintWriter(new File("Scala_RDD_test_parall.csv"))
// Écrire l’en-tête (une seule fois avant la boucle)
pw.println(Seq("methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps").mkString(","))
// test pour différents niveau de parallélisation
for ( n <- test_nb_paralle ) { runCCF_RDD(df, 100000, CCFUtils.ccf_iterate_RDD , n , "yes" , "RDD_scala_iterate_sans_duplicat", pw) }
pw.close() // fermeture du fichier

Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 1, temps écoulé : 11.175 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 2, temps écoulé : 7.709 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 3, temps écoulé : 7.839 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 4, temps écoulé : 7.711 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 5, temps écoulé : 6.445 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 6, temps écoulé : 7.798 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 7, temps écoulé : 6.803 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 8, temps écoulé : 6.722 secondes
Test RDD_scala_iterate_sans_duplicat avec 100000 liens et parallélisation 9, temps écoulé : 7.646 secondes
Test RDD_scala_iterate_sans_duplicat

test_nb_paralle = ArrayBuffer(1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30)
pw = java.io.PrintWriter@25bba867


java.io.PrintWriter@25bba867

### Execution de la function runCCF avec comme variable :
- df : le tableau complet de données Google

- n : nombre d'arrêtes sélectionnées pour le test

- ccf : choix de l'algoritme

- param_parall : nombre de parallélisation

- param_deduplic : "yes ou "no" pour définir si utilisation de CCF-dedup

- method : Legende du test fait

- pw : fichier pour enregistre les résultats du test

In [35]:
// Execution des tests et sauvegarde des résultats 

// choix du nombre du nombre de parallélisation 3 est le meilleur score du fait qu'il s'agit du max possible sur le PC
// mais pour ne pas surcharger la machine au niveau mémoire sélection de 6 parallélisation

// Création du writer (mode écriture) // ATTENTION : ça écrase le fichier si déjà existant
val pw = new PrintWriter(new File("/data/indir/Scala_RDD_3.csv"))
// Écrire l’en-tête (une seule fois avant la boucle)
pw.println(Seq("methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps").mkString(","))

for ( n <- test_vol_data ) { runCCF_RDD(df, n, CCFUtils.ccf_iterate_RDD , 3 , "no" , "RDD_scala_iterate_avec_duplicat", pw) }
for ( n <- test_vol_data ) { runCCF_RDD(df, n, CCFUtils.ccf_iterate_RDD , 3 , "yes" , "RDD_scala_iterate_sans_duplicat", pw) }
for ( n <- test_vol_data ) { runCCF_RDD(df, n, CCF2Utils.ccf_iterate_sort_RDD , 3 , "yes" , "RDD_scala_iterate_sort_sans_duplicat", pw) }
runCCF_RDD(df, df.count().toInt, CCFUtils.ccf_iterate_RDD , 3 , "yes" , "RDD_scala_iterate_sans_duplicat", pw) 
pw.close() // Ne fermer ici si ajout des résultats RDD dans ce même fichier

Test RDD_scala_iterate_avec_duplicat avec 10 liens et parallélisation 3, temps écoulé : 0.226 secondes
Test RDD_scala_iterate_avec_duplicat avec 10 liens et parallélisation 3, temps écoulé : 0.320 secondes
Test RDD_scala_iterate_avec_duplicat avec 10 liens et parallélisation 3, temps écoulé : 0.270 secondes
Test RDD_scala_iterate_avec_duplicat avec 100 liens et parallélisation 3, temps écoulé : 0.308 secondes
Test RDD_scala_iterate_avec_duplicat avec 100 liens et parallélisation 3, temps écoulé : 0.260 secondes
Test RDD_scala_iterate_avec_duplicat avec 100 liens et parallélisation 3, temps écoulé : 0.256 secondes
Test RDD_scala_iterate_avec_duplicat avec 1000 liens et parallélisation 3, temps écoulé : 0.215 secondes
Test RDD_scala_iterate_avec_duplicat avec 1000 liens et parallélisation 3, temps écoulé : 0.238 secondes
Test RDD_scala_iterate_avec_duplicat avec 1000 liens et parallélisation 3, temps écoulé : 0.247 secondes
Test RDD_scala_iterate_avec_duplicat avec 10000 liens et parallé

pw = java.io.PrintWriter@5b3e28d


java.io.PrintWriter@5b3e28d

In [41]:
// lancement individuel pour réalisation de tests 5M
val pw = new PrintWriter(new File("/data/indir/Scala_RDD_max_opti_5M_sort.csv"))
// Écrire l’en-tête (une seule fois avant la boucle)
pw.println(Seq("methode","nb_init_arretes","nb_final_arretes","nb_groupes","nb_iter_CCF","nb_parellele","temps").mkString(","))
runCCF_RDD(df, df.count().toInt, CCF2Utils.ccf_iterate_sort_RDD , 3 , "yes" , "RDD_scala_iterate_sort_sans_duplicat", pw) 
pw.close()

lastException = null


Syntax Error.: 